# Embeddings Visualization Lab
## System Handler — Semana 3 · Reel 11 Support Notebook

Este notebook te ayuda a **ver visualmente** cómo palabras o frases con significado parecido
quedan **cerca en el espacio vectorial**.

### Qué hace
- Genera embeddings con OpenAI
- Reduce dimensiones con PCA
- Los visualiza en **2D** y **3D** con Plotly
- Calcula similitud coseno para comparar significados

> Nota: un embedding real tiene muchas dimensiones.  
> Para visualizarlo, lo proyectamos a 2D o 3D.

## Requisitos

Tu `pyproject.toml` ya incluye:

- `plotly>=6.3.0`
- `scikit-learn>=1.7.2`

Además necesitas tener disponible:
- `openai`
- una variable de entorno `OPENAI_API_KEY`

In [1]:
import os
from typing import List

import numpy as np
import plotly.graph_objects as go
from openai import OpenAI
from sklearn.decomposition import PCA

In [2]:
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "No encontré la variable OPENAI_API_KEY. "
        "Configúrala antes de ejecutar este notebook."
    )

client = OpenAI(api_key=api_key)
print("Cliente OpenAI listo.")

Cliente OpenAI listo.


## 1) Textos de prueba

Puedes cambiar esta lista por palabras, frases o incluso oraciones cortas.
La idea es mezclar grupos semánticos parecidos y diferentes para ver la separación.

In [63]:
texts: List[str] = [
    "ventas",
    "ingresos",
    "revenue",
    "ganancias",
    "dinero",
    "perro",
    "gato",
    "mascota",
    "automóvil",
    "coche",
    "vehículo",
    "transporte",
]
texts

['ventas',
 'ingresos',
 'revenue',
 'ganancias',
 'dinero',
 'perro',
 'gato',
 'mascota',
 'automóvil',
 'coche',
 'vehículo',
 'transporte']

## 2) Generar embeddings

Usamos `text-embedding-3-small` porque es una opción práctica para demos y notebooks.

In [34]:
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts
)

embeddings = [item.embedding for item in response.data]
embedding_matrix = np.array(embeddings, dtype=float)

print("Número de textos:", len(texts))
print("Dimensiones del embedding matrix:", embedding_matrix.shape)

Número de textos: 12
Dimensiones del embedding matrix: (12, 1536)


## 3) Proyección a 2D con PCA

PCA reduce dimensionalidad para que podamos graficar los embeddings.
Esto **no muestra todo el embedding real**, pero sí una proyección útil para entender la idea.

In [46]:
pca_2d = PCA(n_components=2)
reduced_2d = pca_2d.fit_transform(embedding_matrix)

print("Shape 2D:", reduced_2d.shape)
print("Varianza explicada:", pca_2d.explained_variance_ratio_)

Shape 2D: (12, 2)
Varianza explicada: [0.21248717 0.17230646]


In [47]:
fig_2d = go.Figure()

fig_2d.add_trace(
    go.Scatter(
        x=reduced_2d[:, 0],
        y=reduced_2d[:, 1],
        mode="markers+text",
        text=texts,
        textposition="top center",
        marker=dict(size=12),
        hovertemplate="<b>%{text}</b><br>x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>",
    )
)

fig_2d.update_layout(
    title="Embeddings proyectados a 2D",
    xaxis_title="Componente principal 1",
    yaxis_title="Componente principal 2",
    template="plotly_dark",
    height=700,
)

fig_2d.show()

## 4) Proyección a 3D con PCA

Esta vista suele verse muy bien en video porque da más sensación de espacio semántico.

In [48]:
pca_3d = PCA(n_components=3)
reduced_3d = pca_3d.fit_transform(embedding_matrix)

print("Shape 3D:", reduced_3d.shape)
print("Varianza explicada:", pca_3d.explained_variance_ratio_)

Shape 3D: (12, 3)
Varianza explicada: [0.21248717 0.17230646 0.09939274]


In [49]:
fig_3d = go.Figure(
    data=[
        go.Scatter3d(
            x=reduced_3d[:, 0],
            y=reduced_3d[:, 1],
            z=reduced_3d[:, 2],
            mode="markers+text",
            text=texts,
            textposition="top center",
            marker=dict(size=6),
            hovertemplate=(
                "<b>%{text}</b><br>"
                "x=%{x:.3f}<br>y=%{y:.3f}<br>z=%{z:.3f}<extra></extra>"
            ),
        )
    ]
)

fig_3d.update_layout(
    title="Embeddings proyectados a 3D",
    template="plotly_dark",
    height=800,
    scene=dict(
        xaxis_title="PC1",
        yaxis_title="PC2",
        zaxis_title="PC3",
    ),
)

fig_3d.show()

## 5) Similitud coseno

Además de ver los puntos, conviene medir qué tan parecidos son entre sí.

La similitud coseno:
- más cerca de **1.0** → más parecidos
- más cerca de **0.0** → menos relacionados
- negativa → direcciones opuestas en el espacio

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

similarity_matrix = np.zeros((len(texts), len(texts)), dtype=float)

for i in range(len(texts)):
    for j in range(len(texts)):
        similarity_matrix[i, j] = cosine_similarity(embedding_matrix[i], embedding_matrix[j])

similarity_matrix.shape

In [ ]:
heatmap = go.Figure(
    data=go.Heatmap(
        z=similarity_matrix,
        x=texts,
        y=texts,
        hoverongaps=False,
        text=np.round(similarity_matrix, 3),
        texttemplate="%{text}",
    )
)

heatmap.update_layout(
    title="Matriz de similitud coseno",
    template="plotly_dark",
    height=800,
)

heatmap.show()

## 6) Comparaciones puntuales

Esta celda te deja revisar pares específicos, útil para explicar el concepto en cámara.

In [ ]:
pairs = [
    ("ventas", "ingresos"),
    ("ventas", "revenue"),
    ("perro", "gato"),
    ("coche", "vehículo"),
    ("ventas", "perro"),
]

index_by_text = {text: i for i, text in enumerate(texts)}

for left, right in pairs:
    score = similarity_matrix[index_by_text[left], index_by_text[right]]
    print(f"{left:>12}  vs  {right:<12} -> similitud: {score:.4f}")

## 7) Cómo contar esto en tu video

Puedes explicarlo así:

> “Cada palabra o frase se convierte en un punto matemático.”  
> “Y si dos ideas significan algo parecido, esos puntos quedan cerca.”  
> “Eso es lo que hace posible buscar por significado, no solo por texto exacto.”

## Ideas para probar
- Cambiar palabras por frases de negocio
- Mezclar español e inglés
- Comparar conceptos técnicos
- Probar oraciones completas en lugar de palabras sueltas

## 8) Siguiente paso natural

Después de este notebook, el flujo perfecto para la serie sería:

**texto → embedding → búsqueda → contexto → respuesta**

Eso conecta directamente con:
- chunks
- vector databases
- RAG
- Top-K similarity search